[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jiehou-lab/urban-ai/blob/main/notebooks/lab2_clustering.ipynb)

# Lab 2: Clustering Analysis

**Duration:** ~1.5 hours
**TA lead:** Sean
**Course:** Urban AI — AI-Driven Decision Support for Real-World Urban Challenges (MSU AI-Ready Initiative)

## Learning goals
- Standardize and cluster census-tract indicators with k-means.
- Use silhouette score to compare different numbers of clusters (k).
- See directly how changing k changes which tracts get grouped together.
- Produce a neighborhood typology map with a documented justification for the chosen k.


## Before you start: Track A vs. Track B

Every Urban AI lab has two tracks. Both produce the **same artifact**: **Neighborhood typology map + k justification**.

- **Track A — No code (default).** Use an interactive cluster explorer with sliders. No installation, no Python required — use this track if you would rather click through a web tool.
- **Track B — Colab (this notebook).** Run k-means + silhouette analysis on census-tract indicators, changing k. You will run pre-written cells and change only the parameters marked `# ▶ CHANGE ME`. You will never need to write code from scratch.

Both tracks end with the same 4 reflection prompts (the last cell of this notebook).


## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

RNG = np.random.default_rng(11)
plt.rcParams["font.size"] = 12
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12
print("Setup complete.")

## 2. Load the data: census-tract indicators for one metro

> **Synthetic-but-realistic data.** The dataset below is generated in this notebook with a fixed
> random seed so the lab runs the same way for everyone, completely offline. It is built to look and
> behave like real urban data, but it is not real. To swap in real data for your own city, instructors
> can replace the data-generation cell with a download/load from a real source such as:
- US Census Bureau American Community Survey (ACS) 5-year estimates, via data.census.gov or the Census API
- City/county GIS open data portals for tract boundary shapefiles
- EPA EJScreen or CDC/ATSDR Social Vulnerability Index for supplementary equity indicators


In [ ]:
n_tracts = 120

# "True" underlying neighborhood archetypes used only to generate realistic synthetic data
archetypes = {
    "Urban Core":          dict(median_income=42000, pct_renter=0.75, pct_over65=0.10, density=9000, pct_no_vehicle=0.35, home_value=210000),
    "Inner Suburb":        dict(median_income=68000, pct_renter=0.35, pct_over65=0.18, density=3500, pct_no_vehicle=0.08, home_value=260000),
    "Outer Suburb":        dict(median_income=95000, pct_renter=0.15, pct_over65=0.15, density=1200, pct_no_vehicle=0.02, home_value=340000),
    "Legacy/Disinvested":  dict(median_income=31000, pct_renter=0.55, pct_over65=0.22, density=4200, pct_no_vehicle=0.30, home_value=95000),
    "College Town":        dict(median_income=38000, pct_renter=0.80, pct_over65=0.05, density=6000, pct_no_vehicle=0.25, home_value=180000),
}
archetype_names = list(archetypes.keys())
tract_archetype = RNG.choice(archetype_names, size=n_tracts, p=[0.25, 0.25, 0.20, 0.15, 0.15])
lon_offset = {"Urban Core": 0.0, "Inner Suburb": 0.3, "Outer Suburb": 0.6,
              "Legacy/Disinvested": -0.3, "College Town": -0.6}


def noisy(val, pct_noise=0.15):
    return val * (1 + RNG.normal(0, pct_noise))


rows = []
for i in range(n_tracts):
    a = archetypes[tract_archetype[i]]
    rows.append({
        "tract_id": f"T{100+i}",
        "median_income": max(15000, noisy(a["median_income"])),
        "pct_renter": float(np.clip(noisy(a["pct_renter"]), 0, 1)),
        "pct_over65": float(np.clip(noisy(a["pct_over65"]), 0, 1)),
        "population_density": max(100, noisy(a["density"])),
        "pct_no_vehicle": float(np.clip(noisy(a["pct_no_vehicle"]), 0, 1)),
        "median_home_value": max(30000, noisy(a["home_value"])),
        "lon": RNG.uniform(-1, 1) * 0.15 + lon_offset[tract_archetype[i]],
        "lat": RNG.uniform(-1, 1) * 0.15,
    })
tracts = pd.DataFrame(rows)
tracts.head()

### Standardize the indicators
Income is measured in tens of thousands of dollars and vehicle-access rate is measured in fractions of 1 -- without standardizing, k-means would let income dominate purely because of its larger numbers, not because it matters more.

In [ ]:
# ▶ CHANGE ME: remove an indicator (e.g. "population_density") to see how the typology changes
FEATURES = ["median_income", "pct_renter", "pct_over65", "population_density",
            "pct_no_vehicle", "median_home_value"]
X = tracts[FEATURES].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Scaled feature means (should be close to 0):", X_scaled.mean(axis=0).round(2))
print("Scaled feature std devs (should be close to 1):", X_scaled.std(axis=0).round(2))

### Run k-means with a starting number of clusters
k-means groups tracts that are similar across all six indicators at once -- something no single map or chart can show directly.

In [ ]:
# ▶ CHANGE ME: try 3, 5, or 6 clusters instead of 4
K_DEFAULT = 4

km = KMeans(n_clusters=K_DEFAULT, random_state=42, n_init=10)
tracts["cluster"] = km.fit_predict(X_scaled)
sil = silhouette_score(X_scaled, tracts["cluster"])
print(f"k={K_DEFAULT}: silhouette score = {sil:.3f} (higher is better, max is 1.0)")

### Map the clusters
We do not have GIS software here, so we plot each tract's synthetic location colored by cluster -- a quick stand-in for a real neighborhood-typology map.

In [ ]:
fig_map, ax = plt.subplots(figsize=(7, 6))
scatter = ax.scatter(tracts["lon"], tracts["lat"], c=tracts["cluster"], cmap="tab10", s=60)
ax.set_title(f"Neighborhood Typology Map (k={K_DEFAULT} clusters)")
ax.set_xlabel("Synthetic longitude (relative)")
ax.set_ylabel("Synthetic latitude (relative)")
legend1 = ax.legend(*scatter.legend_elements(), title="Cluster")
ax.add_artist(legend1)
plt.tight_layout()
plt.show()

### Profile each cluster
Averaging the indicators within each cluster lets us name the typologies (e.g., "lower-income, high-density, high-transit-dependence") instead of just calling them Cluster 0, 1, 2...

In [ ]:
profile = tracts.groupby("cluster")[FEATURES].mean().round(1)
profile["n_tracts"] = tracts.groupby("cluster").size()
profile

### Responsible AI check: does k change who gets grouped with whom?
There is no single "correct" number of clusters. As we change k, some tracts' neighbors in the typology change completely -- which matters if this typology is used to target funding or programs.

In [ ]:
K_RANGE = [3, 4, 5, 6]
sil_scores = {}
membership = pd.DataFrame({"tract_id": tracts["tract_id"]})
for k in K_RANGE:
    km_k = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km_k.fit_predict(X_scaled)
    membership[f"k={k}"] = labels
    sil_scores[k] = silhouette_score(X_scaled, labels)

print("Silhouette score by k:")
for k, s in sil_scores.items():
    print(f"  k={k}: {s:.3f}")

# A handful of sample tracts -- watch how their cluster number (and therefore their "neighbors"
# in the typology) shifts as k changes.
sample_tracts = tracts["tract_id"].iloc[[0, 20, 40, 60, 80]].tolist()
membership[membership["tract_id"].isin(sample_tracts)]

## Experiment

Try changing the parameters marked `# ▶ CHANGE ME` in the next cell(s) and re-run. Specifically, try:

1. Change `K_DEFAULT` above to 3, 5, or 6 and re-run the map cell -- which tracts change color?
2. Remove `population_density` from `FEATURES` and re-run -- does the typology still separate Urban Core from Outer Suburb?
3. Compare the silhouette scores across `K_RANGE` -- is the highest score always the most useful k for a planner?


## Artifact: neighborhood typology map + k justification

In [ ]:
best_k = max(sil_scores, key=sil_scores.get)
final_km = KMeans(n_clusters=best_k, random_state=42, n_init=10)
tracts["cluster_final"] = final_km.fit_predict(X_scaled)

fig_final_map, ax = plt.subplots(figsize=(7, 6))
scatter = ax.scatter(tracts["lon"], tracts["lat"], c=tracts["cluster_final"], cmap="tab10", s=60)
ax.set_title(f"ARTIFACT: Neighborhood Typology Map (k={best_k})")
ax.set_xlabel("Synthetic longitude (relative)")
ax.set_ylabel("Synthetic latitude (relative)")
legend1 = ax.legend(*scatter.legend_elements(), title="Cluster")
ax.add_artist(legend1)
plt.tight_layout()
plt.savefig("lab2_typology_map.png", dpi=150, bbox_inches="tight")
plt.show()

justification = (
    f"We evaluated k in {K_RANGE} using silhouette score. k={best_k} scored highest "
    f"({sil_scores[best_k]:.3f}), so we recommend k={best_k} clusters for this metro's neighborhood "
    f"typology. However, cluster membership is sensitive to the choice of k (see the membership table "
    f"above), so this typology should be reviewed by a local planner before being used to target "
    f"programs or funding."
)
print(justification)

final_profile = tracts.groupby("cluster_final")[FEATURES].mean().round(1)
final_profile.to_csv("lab2_typology_profile.csv")
with open("lab2_k_justification.txt", "w") as f:
    f.write(justification)
print("\nSaved artifacts: lab2_typology_map.png, lab2_typology_profile.csv, lab2_k_justification.txt")

## Reflect (answer in your own words — this is part of your mini-task)

1. What did the tool assume?
2. Who is missing from this data?
3. What would change your recommendation?
4. What must a human verify before this is used?
